12/09/2026
First version

In [1]:
from io import TextIOWrapper
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import numpy as np
import pandas as pd
import zstandard as zstd
from sklearn.tree import DecisionTreeClassifier


KeyboardInterrupt: 

In [ ]:
from io import StringIO

PGN_ZST_PATH = Path("DB/lichess_db_standard_rated_2013-01.pgn.zst")


def partidas_en_stream(ruta: Path):
    """Genera partidas PGN una a una desde un archivo .pgn.zst."""
    with ruta.open("rb") as archivo_comprimido:
        descompresor = zstd.ZstdDecompressor()
        with descompresor.stream_reader(archivo_comprimido) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")
            while partida := chess.pgn.read_game(flujo_texto):
                yield partida


def partida_en_indice(ruta: Path, indice: int):
    """Devuelve la partida con índice cero-based sin cargar todas las partidas."""
    if indice < 0:
        raise ValueError("El índice debe ser mayor o igual que cero")

    for indice_actual, partida in enumerate(partidas_en_stream(ruta)):
        if indice_actual == indice:
            return partida

    raise IndexError(f"No existe una partida con índice {indice}")


def partida_para_prueba(ruta: Path, indice: int):
    """Localiza una partida sin analizar el PGN de las partidas anteriores."""
    if indice < 0:
        raise ValueError("El índice debe ser mayor o igual que cero")

    partida_actual = -1
    lineas_partida = []

    with ruta.open("rb") as archivo_comprimido:
        descompresor = zstd.ZstdDecompressor()
        with descompresor.stream_reader(archivo_comprimido) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")

            for linea in flujo_texto:
                if linea.startswith("[Event "):
                    partida_actual += 1
                    if partida_actual > indice:
                        break
                    if partida_actual == indice:
                        lineas_partida = [linea]
                    continue

                if partida_actual == indice:
                    lineas_partida.append(linea)

    if partida_actual < indice or not lineas_partida:
        raise IndexError(f"No existe una partida con índice {indice}")

    partida = chess.pgn.read_game(StringIO("".join(lineas_partida)))
    if partida is None:
        raise ValueError(f"No se pudo interpretar la partida con índice {indice}")

    return partida


# INDICE_PARTIDA = 10
# partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
# resultado_final = partida_seleccionada.headers.get("Result", "*")
# tablero = partida_seleccionada.board()
# movimientos_por_numero = {}

# for movimiento in partida_seleccionada.mainline_moves():
#     numero_movimiento = tablero.fullmove_number
#     notacion = tablero.san(movimiento)
#     movimientos_por_numero.setdefault(numero_movimiento, []).append(notacion)
#     tablero.push(movimiento)

# ultimos_10_movimientos = [
#     f"{numero}. {' '.join(movimientos)}"
#     for numero, movimientos in list(movimientos_por_numero.items())[-10:]
# ]

# print("Índice:", INDICE_PARTIDA)
# print("Resultado final:", resultado_final)
# print("Últimos 10 movimientos:")
# print(" ".join(ultimos_10_movimientos))

In [ ]:
RESULTADO_A_CLASE = {
    "1-0": 0,
    "0-1": 1,
    "1/2-1/2": 2,
}

CARACTERISTICAS = [
    "reina_blancas",
    "reina_negras",
    "blancas_enrocadas",
    "negras_enrocadas",
    "dos_alfiles_blancas",
    "dos_alfiles_negras",
    "peon_pasado_blancas",
    "peon_pasado_negras",
    "mas_piezas_blancas",
    "mas_piezas_negras",
    "mas_peones_blancas",
    "mas_peones_negras",
    "torre_adelantada_blancas",
    "torre_adelnatada_negras",
 ]


def tiene_peon_pasado(tablero: chess.Board, color: chess.Color) -> bool:
    """Indica si el color tiene al menos un peón pasado."""
    peones = tablero.pieces(chess.PAWN, color)
    peones_rivales = tablero.pieces(chess.PAWN, not color)

    for casilla in peones:
        archivo = chess.square_file(casilla)
        rango = chess.square_rank(casilla)
        tiene_peon_rival_delante = any(
            abs(chess.square_file(casilla_rival) - archivo) <= 1
            and (
                chess.square_rank(casilla_rival) > rango
                if color == chess.WHITE
                else chess.square_rank(casilla_rival) < rango
            )
            for casilla_rival in peones_rivales
        )
        if not tiene_peon_rival_delante:
            return True

    return False


def tiene_torre_adelantada(tablero: chess.Board, color: chess.Color) -> bool:
    """Indica si hay una torre en las dos filas más cercanas al rival."""
    filas_adelantadas = {6, 7} if color == chess.WHITE else {0, 1}
    return any(
        chess.square_rank(casilla) in filas_adelantadas
        for casilla in tablero.pieces(chess.ROOK, color)
    )


def estado_tablero(tablero: chess.Board, blancas_enrocadas: bool, negras_enrocadas: bool):
    """Extrae las características relevantes de una posición."""
    piezas_blancas = sum(
        len(tablero.pieces(tipo, chess.WHITE))
        for tipo in range(chess.PAWN, chess.KING)
    )
    piezas_negras = sum(
        len(tablero.pieces(tipo, chess.BLACK))
        for tipo in range(chess.PAWN, chess.KING)
    )
    peones_blancos = len(tablero.pieces(chess.PAWN, chess.WHITE))
    peones_negros = len(tablero.pieces(chess.PAWN, chess.BLACK))

    return {
        "reina_blancas": int(bool(tablero.pieces(chess.QUEEN, chess.WHITE))),
        "reina_negras": int(bool(tablero.pieces(chess.QUEEN, chess.BLACK))),
        "blancas_enrocadas": int(blancas_enrocadas),
        "negras_enrocadas": int(negras_enrocadas),
        "dos_alfiles_blancas": int(len(tablero.pieces(chess.BISHOP, chess.WHITE)) >= 2),
        "dos_alfiles_negras": int(len(tablero.pieces(chess.BISHOP, chess.BLACK)) >= 2),
        "peon_pasado_blancas": int(tiene_peon_pasado(tablero, chess.WHITE)),
        "peon_pasado_negras": int(tiene_peon_pasado(tablero, chess.BLACK)),
        "mas_piezas_blancas": int(piezas_blancas > piezas_negras),
        "mas_piezas_negras": int(piezas_negras > piezas_blancas),
        "mas_peones_blancas": int(peones_blancos > peones_negros),
        "mas_peones_negras": int(peones_negros > peones_blancos),
        "torre_adelantada_blancas": int(tiene_torre_adelantada(tablero, chess.WHITE)),
        "torre_adelnatada_negras": int(tiene_torre_adelantada(tablero, chess.BLACK)),
    }


def partida_a_dataframe(partida: chess.pgn.Game, movimientos_desde_final: int = 10):
    """Devuelve una fila con el estado 10 jugadas completas antes del final."""
    if movimientos_desde_final < 0:
        raise ValueError("movimientos_desde_final debe ser mayor o igual que cero")

    resultado = partida.headers.get("Result", "*")
    if resultado not in RESULTADO_A_CLASE:
        raise ValueError(f"Resultado PGN no válido para clasificación: {resultado}")

    movimientos = list(partida.mainline_moves())
    tablero = partida.board()
    estados = []
    blancas_enrocadas = False
    negras_enrocadas = False

    estados.append(estado_tablero(tablero, blancas_enrocadas, negras_enrocadas))

    for movimiento in movimientos:
        if tablero.is_castling(movimiento):
            if tablero.turn == chess.WHITE:
                blancas_enrocadas = True
            else:
                negras_enrocadas = True

        tablero.push(movimiento)
        estados.append(estado_tablero(tablero, blancas_enrocadas, negras_enrocadas))

    plies_desde_final = movimientos_desde_final * 2
    indice_estado = max(0, len(estados) - 1 - plies_desde_final)
    caracteristicas = estados[indice_estado]
    caracteristicas["y"] = RESULTADO_A_CLASE[resultado]

    return pd.DataFrame([caracteristicas], columns=CARACTERISTICAS + ["y"])


# INDICE_PARTIDA = 20
# partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
# datos_partida = partida_a_dataframe(partida_seleccionada)
# X = datos_partida[CARACTERISTICAS]
# y = datos_partida["y"]

# print("Índice:", INDICE_PARTIDA)
# print("Resultado:", partida_seleccionada.headers["Result"])
# print("Forma de X:", X.shape)
# print("Forma de y:", y.shape)
# datos_partida

In [ ]:
def extrae_caracteristicas(ruta: Path, numero_observaciones: int = 2000):
    """Crea un DataFrame hasta alcanzar el número de observaciones solicitado."""
    if numero_observaciones < 0:
        raise ValueError("numero_observaciones debe ser mayor o igual que cero")

    datos_partidas = []
    claves_observaciones = set()

    if numero_observaciones == 0:
        return pd.DataFrame(columns=CARACTERISTICAS + ["y"])

    for partida in partidas_en_stream(ruta):
        datos_partida = partida_a_dataframe(partida)
        claves_partida = datos_partida[CARACTERISTICAS].apply(tuple, axis=1)
        nuevas_observaciones = ~claves_partida.isin(claves_observaciones)

        if nuevas_observaciones.any():
            datos_partidas.append(datos_partida.loc[nuevas_observaciones])
            claves_observaciones.update(claves_partida[nuevas_observaciones])

        if len(claves_observaciones) >= numero_observaciones:
            break

    if not datos_partidas:
        return pd.DataFrame(columns=CARACTERISTICAS + ["y"])

    return pd.concat(datos_partidas, ignore_index=True).head(numero_observaciones)


datos_observaciones = extrae_caracteristicas(PGN_ZST_PATH, 1500)
datos_observaciones

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_observaciones = datos_observaciones[CARACTERISTICAS]
y_observaciones = datos_observaciones["y"]

X_train, X_validation, y_train, y_validation = train_test_split(
    X_observaciones,
    y_observaciones,
    test_size=0.2,
    random_state=42,
    stratify=y_observaciones,
)

arbol_decision = DecisionTreeClassifier(random_state=42)
arbol_decision.fit(X_train, y_train)

predicciones_validation = arbol_decision.predict(X_validation)
precision_validation = accuracy_score(y_validation, predicciones_validation)

print(f"Precisión en validación: {precision_validation:.2%}")
precision_validation

In [ ]:
INDICE_PARTIDA_PRUEBA = 35125
partida_prueba = partida_para_prueba(PGN_ZST_PATH, INDICE_PARTIDA_PRUEBA)
resultado_real = partida_prueba.headers.get("Result", "*")

if resultado_real not in RESULTADO_A_CLASE:
    raise ValueError(f"Resultado PGN no válido: {resultado_real}")

movimientos = list(partida_prueba.mainline_moves())
tablero = partida_prueba.board()
resultados_prueba = []
blancas_enrocadas = False
negras_enrocadas = False

# Solo se evalúan posiciones después de un movimiento completo (dos plies).
# La última posición evaluada deja al menos 10 movimientos completos hasta el final.
for inicio in range(0, len(movimientos) - 21, 2):
    for movimiento in movimientos[inicio:inicio + 2]:
        if tablero.is_castling(movimiento):
            if tablero.turn == chess.WHITE:
                blancas_enrocadas = True
            else:
                negras_enrocadas = True
        tablero.push(movimiento)

    caracteristicas = estado_tablero(tablero, blancas_enrocadas, negras_enrocadas)
    X_posicion = pd.DataFrame([caracteristicas], columns=CARACTERISTICAS)
    prediccion = int(arbol_decision.predict(X_posicion)[0])

    resultados_prueba.append(
        {
            "movimiento_completo": tablero.fullmove_number - 1,
            "prediccion": prediccion,
            "resultado_real": RESULTADO_A_CLASE[resultado_real],
            "acierto": prediccion == RESULTADO_A_CLASE[resultado_real],
        }
    )

resultados_prueba = pd.DataFrame(resultados_prueba)

if resultados_prueba.empty:
    raise ValueError("La partida no tiene suficientes movimientos para esta evaluación")

aciertos_prueba = resultados_prueba["acierto"].sum()
numero_predicciones = len(resultados_prueba)
precision_prueba = resultados_prueba["acierto"].mean()

print(f"Partida evaluada: {INDICE_PARTIDA_PRUEBA}")
print(f"Resultado real: {resultado_real}")
print(f"Predicciones correctas: {aciertos_prueba}/{numero_predicciones}")
print(f"Precisión en la partida: {precision_prueba:.2%}")

resultados_prueba

In [ ]:
from sklearn.ensemble import RandomForestClassifier

bosque_aleatorio = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
bosque_aleatorio.fit(X_train, y_train)

predicciones_bosque = bosque_aleatorio.predict(X_validation)
precision_bosque = accuracy_score(y_validation, predicciones_bosque)

print(f"Precisión del árbol de decisión: {precision_validation:.2%}")
print(f"Precisión del random forest: {precision_bosque:.2%}")
print(f"Mejora del random forest: {(precision_bosque - precision_validation):+.2%}")

precision_bosque

In [ ]:
import joblib

modelo_random_forest = {
    "modelo": bosque_aleatorio,
    "caracteristicas": CARACTERISTICAS,
    "resultado_a_clase": RESULTADO_A_CLASE,
    "version_modelo": 1,
}

ruta_modelo = Path("modelo_chess_random_forest.joblib")
joblib.dump(modelo_random_forest, ruta_modelo)

print(f"Modelo guardado en: {ruta_modelo}")
print(f"Tamaño del archivo: {ruta_modelo.stat().st_size / 1024:.1f} KB")

In [ ]:
INDICE_PARTIDA_PREDICCION = 35125
partida_prediccion = partida_para_prueba(PGN_ZST_PATH, INDICE_PARTIDA_PREDICCION)
resultado_real_prediccion = partida_prediccion.headers.get("Result", "*")

if resultado_real_prediccion not in RESULTADO_A_CLASE:
    raise ValueError(
        f"Resultado PGN no válido para clasificación: {resultado_real_prediccion}"
    )

movimientos_prediccion = list(partida_prediccion.mainline_moves())
tablero_prediccion = partida_prediccion.board()
resultados_prediccion = []
blancas_enrocadas_prediccion = False
negras_enrocadas_prediccion = False

clase_a_resultado = {
    clase: resultado for resultado, clase in RESULTADO_A_CLASE.items()
}

# Se predice después de cada movimiento completo hasta dejar 10 movimientos al final.
for inicio in range(0, len(movimientos_prediccion) - 21, 2):
    for movimiento in movimientos_prediccion[inicio:inicio + 2]:
        if tablero_prediccion.is_castling(movimiento):
            if tablero_prediccion.turn == chess.WHITE:
                blancas_enrocadas_prediccion = True
            else:
                negras_enrocadas_prediccion = True
        tablero_prediccion.push(movimiento)

    caracteristicas_prediccion = estado_tablero(
        tablero_prediccion,
        blancas_enrocadas_prediccion,
        negras_enrocadas_prediccion,
    )
    X_posicion_prediccion = pd.DataFrame(
        [caracteristicas_prediccion],
        columns=CARACTERISTICAS,
    )
    prediccion = int(bosque_aleatorio.predict(X_posicion_prediccion)[0])
    resultado_predicho = clase_a_resultado[prediccion]

    resultados_prediccion.append(
        {
            "movimiento_completo": tablero_prediccion.fullmove_number - 1,
            "prediccion": resultado_predicho,
            "resultado_real": resultado_real_prediccion,
            "acierto": resultado_predicho == resultado_real_prediccion,
        }
    )

resultados_prediccion = pd.DataFrame(resultados_prediccion)

if resultados_prediccion.empty:
    raise ValueError(
        "La partida no tiene suficientes movimientos para esta evaluación"
    )

print(f"Índice de la partida: {INDICE_PARTIDA_PREDICCION}")
print(f"Resultado real: {resultado_real_prediccion}")
print(
    f"Predicciones correctas: "
    f"{resultados_prediccion['acierto'].sum()}/"
    f"{len(resultados_prediccion)}"
)
print(
    f"Precisión del random forest en la partida: "
    f"{resultados_prediccion['acierto'].mean():.2%}"
)

resultados_prediccion